# 📘 DateTime & Timezone Handling for Data Engineering
## Databricks Notebook — Time is the Hardest Problem

**What you'll learn:**
- Python datetime module: creation, arithmetic, formatting, parsing
- Timezone handling: UTC standard, conversions, DST edge cases
- Epoch timestamps: seconds vs milliseconds detection
- Parsing messy dates from multiple source systems
- PySpark datetime functions
- Date partitioning strategies
- Complete date dimension generator

**Key rules:**
- ALWAYS store timestamps in UTC in your data lake
- NEVER use naive datetimes in production
- Use ISO 8601 format as the standard string representation

**Prerequisites:** Notebooks 01-13

---

---
# 📗 Section 1: Why DateTime is Hard in DE

**The problems:**
1. **Timezone hell:** An order placed at "3 PM" — 3 PM WHERE?
2. **DST transitions:** 2:00 AM doesn't exist in spring, 1:00 AM happens twice in fall
3. **Format chaos:** "01/02/03" — Jan 2, 2003? Feb 1, 2003? Jan 2, 1903?
4. **Epoch confusion:** Is `1700000000` seconds or milliseconds?
5. **Late-arriving data:** Event happened yesterday but arrived today

**The solution:** Store everything in UTC. Convert to local only for display.

---
# 📗 Section 2: Python datetime Module

In [0]:
from datetime import datetime, date, time, timedelta, timezone

# Creating datetimes
now = datetime.now()                          # Local time (naive — BAD for production)
utc_now = datetime.now(timezone.utc)          # UTC (aware — GOOD)
specific = datetime(2024, 6, 15, 10, 30, 0)  # Specific datetime

print(f"Local now (naive): {now}")
print(f"UTC now (aware):   {utc_now}")
print(f"Specific:          {specific}")
print(f"Is aware? {utc_now.tzinfo is not None}")

In [0]:
# Arithmetic with timedelta
from datetime import timedelta

start = datetime(2024, 6, 1, 8, 0, 0)
end = start + timedelta(days=30, hours=4, minutes=15)
duration = end - start

print(f"Start: {start}")
print(f"End:   {end}")
print(f"Duration: {duration}")
print(f"Duration in hours: {duration.total_seconds() / 3600:.1f}")

# Business use: calculate SLA deadline
ingestion_time = datetime(2024, 6, 15, 3, 14, 22)
sla_deadline = ingestion_time + timedelta(hours=3)
print(f"\nIngested at: {ingestion_time}")
print(f"SLA deadline (3h): {sla_deadline}")

In [0]:
# Formatting (strftime) and Parsing (strptime)
dt = datetime(2024, 6, 15, 14, 30, 45)

# Format to string
print("Formatting:")
print(f"  ISO:      {dt.isoformat()}")
print(f"  US:       {dt.strftime('%m/%d/%Y %I:%M %p')}")
print(f"  EU:       {dt.strftime('%d-%m-%Y %H:%M')}")
print(f"  Compact:  {dt.strftime('%Y%m%d_%H%M%S')}")
print(f"  Partition:{dt.strftime('year=%Y/month=%m/day=%d')}")

# Parse from string
parsed = datetime.strptime("2024-06-15 14:30:45", "%Y-%m-%d %H:%M:%S")
print(f"\nParsed: {parsed}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Parse Multiple Date Formats
# ============================================
# Parse each of these strings into a datetime object:
date_strings = [
    "2024-06-15T10:30:00Z",       # ISO 8601 with Z
    "Jun 15, 2024 10:30 AM",      # US verbose
    "15/06/2024 10:30",           # EU format
    "20240615",                    # Compact (no time)
    "1718444400",                  # Epoch seconds
]
# Print each parsed result
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
from datetime import datetime, timezone

formats = [
    ("%Y-%m-%dT%H:%M:%SZ", "ISO 8601"),
    ("%b %d, %Y %I:%M %p", "US verbose"),
    ("%d/%m/%Y %H:%M", "EU format"),
    ("%Y%m%d", "Compact"),
]

date_strings = [
    "2024-06-15T10:30:00Z",
    "Jun 15, 2024 10:30 AM",
    "15/06/2024 10:30",
    "20240615",
    "1718444400",
]

for i, (ds, (fmt, name)) in enumerate(zip(date_strings[:4], formats)):
    parsed = datetime.strptime(ds, fmt)
    print(f"  {name}: '{ds}' → {parsed}")

# Epoch
epoch_dt = datetime.fromtimestamp(int(date_strings[4]), tz=timezone.utc)
print(f"  Epoch: '{date_strings[4]}' → {epoch_dt}")

---
# 📗 Section 3: Timezone Handling

**Rule:** Store in UTC. Convert to local only for display.

**Naive** = no timezone info (dangerous)
**Aware** = has timezone info (safe)

In [0]:
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo  # Python 3.9+ (preferred over pytz)

# Create timezone-aware datetimes
utc_time = datetime(2024, 6, 15, 14, 0, 0, tzinfo=timezone.utc)
eastern = utc_time.astimezone(ZoneInfo("America/New_York"))
london = utc_time.astimezone(ZoneInfo("Europe/London"))
tokyo = utc_time.astimezone(ZoneInfo("Asia/Tokyo"))
india = utc_time.astimezone(ZoneInfo("Asia/Kolkata"))

print(f"UTC:     {utc_time.strftime('%Y-%m-%d %H:%M %Z')}")
print(f"New York:{eastern.strftime('%Y-%m-%d %H:%M %Z')} (UTC-4 in summer)")
print(f"London:  {london.strftime('%Y-%m-%d %H:%M %Z')} (UTC+1 in summer)")
print(f"Tokyo:   {tokyo.strftime('%Y-%m-%d %H:%M %Z')} (UTC+9, no DST)")
print(f"India:   {india.strftime('%Y-%m-%d %H:%M %Z')} (UTC+5:30, no DST)")

In [0]:
# Converting local times to UTC (the ingestion pattern)
# Scenario: orders arrive with local timestamps from different regions
local_orders = [
    ("New York", "2024-06-15 10:30:00", "America/New_York"),
    ("London", "2024-06-15 15:30:00", "Europe/London"),
    ("Tokyo", "2024-06-16 00:30:00", "Asia/Tokyo"),
    ("Mumbai", "2024-06-15 20:00:00", "Asia/Kolkata"),
]

print("Converting local times to UTC:")
for city, ts_str, tz_name in local_orders:
    local_dt = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S")
    aware_local = local_dt.replace(tzinfo=ZoneInfo(tz_name))
    utc_dt = aware_local.astimezone(timezone.utc)
    print(f"  {city:<10} {ts_str} {tz_name:<20} → UTC: {utc_dt.strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
# DST edge case: Spring forward (2 AM doesn't exist!)
from zoneinfo import ZoneInfo
from datetime import datetime

# March 10, 2024: US clocks spring forward at 2:00 AM → 3:00 AM
# 2:30 AM doesn't exist!
try:
    # This time doesn't exist (skipped by DST)
    ambiguous = datetime(2024, 3, 10, 2, 30, 0, tzinfo=ZoneInfo("America/New_York"))
    print(f"Spring forward: {ambiguous} → {ambiguous.astimezone(ZoneInfo('UTC'))}")
    print("  Note: zoneinfo adjusts automatically (moves to 3:30 AM)")
except Exception as e:
    print(f"  Error: {e}")

# Fall back: Nov 3, 2024 — 1:30 AM happens TWICE
# fold=0 means first occurrence, fold=1 means second
first_130 = datetime(2024, 11, 3, 1, 30, 0, tzinfo=ZoneInfo("America/New_York"), fold=0)
second_130 = datetime(2024, 11, 3, 1, 30, 0, tzinfo=ZoneInfo("America/New_York"), fold=1)
print(f"\nFall back (1:30 AM first):  {first_130} → UTC: {first_130.astimezone(ZoneInfo('UTC')).strftime('%H:%M')}")
print(f"Fall back (1:30 AM second): {second_130} → UTC: {second_130.astimezone(ZoneInfo('UTC')).strftime('%H:%M')}")

---
# 📗 Section 4: Epoch Timestamp Handling

| Precision | Digits | Example | Source |
|---|---|---|---|
| Seconds | 10 | 1718444400 | Unix, Python |
| Milliseconds | 13 | 1718444400000 | JavaScript, Java |
| Microseconds | 16 | 1718444400000000 | Some databases |

In [0]:
from datetime import datetime, timezone

# Auto-detect epoch precision
def parse_epoch(value):
    s = str(int(value))
    if len(s) <= 10:
        return datetime.fromtimestamp(int(value), tz=timezone.utc), "seconds"
    elif len(s) <= 13:
        return datetime.fromtimestamp(int(value) / 1000, tz=timezone.utc), "milliseconds"
    else:
        return datetime.fromtimestamp(int(value) / 1_000_000, tz=timezone.utc), "microseconds"

# Test
epochs = [1718444400, 1718444400000, 1718444400000000]
for ep in epochs:
    dt, precision = parse_epoch(ep)
    print(f"  {ep:<20} → {dt.isoformat()} ({precision})")

---
# 📗 Section 5: Parsing Messy Dates

In [0]:
from dateutil import parser as dateutil_parser
import pandas as pd

# dateutil handles almost any format
messy_dates = [
    "Jan 5, 2024",
    "5-Jan-2024",
    "2024/01/05",
    "20240105",
    "2024-01-05T10:30:00.000Z",
    "1/5/24",
    "Saturday, January 5th, 2024",
]

print("dateutil.parser.parse() — flexible parsing:")
for ds in messy_dates:
    try:
        parsed = dateutil_parser.parse(ds)
        print(f"  '{ds}' → {parsed.date()}")
    except Exception as e:
        print(f"  '{ds}' → ERROR: {e}")

In [0]:
# pandas batch parsing (faster for known formats)
import pandas as pd

# Mixed formats in a column
date_column = pd.Series([
    "2024-01-15", "2024-02-20", "invalid", "2024-03-10", None, "2024-04-05"
])

# errors='coerce' turns invalid → NaT (Not a Time)
parsed = pd.to_datetime(date_column, errors="coerce")
print("Batch parsing with errors='coerce':")
print(parsed)
print(f"\nNaT count: {parsed.isna().sum()}")

---
# 📗 Section 6: PySpark DateTime Functions

In [0]:
from pyspark.sql.functions import *

spark.sql("USE techmart_dw")
orders = spark.table("orders")

# DateTime extraction and formatting
dated = (orders
    .withColumn("order_year", year("order_date"))
    .withColumn("order_month", month("order_date"))
    .withColumn("order_dow", dayofweek("order_date"))
    .withColumn("days_since_order", datediff(current_date(), col("order_date")))
    .withColumn("formatted", date_format("order_date", "MMM dd, yyyy"))
    .withColumn("epoch_seconds", unix_timestamp("order_date"))
)
dated.select("order_id", "order_date", "order_year", "order_month", "days_since_order", "epoch_seconds").show(5)

In [0]:
# Time-based windowed aggregation
from pyspark.sql.functions import *

orders = spark.table("orders")

# Weekly revenue using window()
weekly = (orders
    .filter("status = 'completed'")
    .withColumn("order_ts", col("order_date").cast("timestamp"))
    .groupBy(window("order_ts", "7 days"))
    .agg(
        count("*").alias("order_count"),
        round(sum("total_amount"), 2).alias("weekly_revenue")
    )
    .select(
        col("window.start").alias("week_start"),
        col("window.end").alias("week_end"),
        "order_count", "weekly_revenue"
    )
    .orderBy("week_start")
)
weekly.show(10)

---
# 📗 Section 7: Partitioning by Date

**Rule of thumb:** Each partition should have 128MB-1GB of data.

| Data Volume | Partition By | Result |
|---|---|---|
| < 1GB/day | month | ~30 partitions/year |
| 1-10GB/day | day | ~365 partitions/year |
| > 10GB/day | hour | ~8760 partitions/year |

In [0]:
%sql
-- Partition website_events by date
USE techmart_dw;
DROP TABLE IF EXISTS website_events_by_date;
CREATE TABLE website_events_by_date
PARTITIONED BY (event_date)
AS SELECT * FROM website_events;

-- Demonstrate partition pruning
EXPLAIN SELECT COUNT(*) FROM website_events_by_date WHERE event_date = '2024-03-15';

---
# 🚀 Section 8: Mini Projects

## Project 1: Timezone-Aware Event Processing

In [0]:
# Process events from multiple timezones → normalize to UTC
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import pandas as pd

# Simulate events from 5 timezones
events = [
    {"event_id": 1, "local_time": "2024-06-15 09:00:00", "timezone": "America/New_York", "event_type": "purchase"},
    {"event_id": 2, "local_time": "2024-06-15 14:30:00", "timezone": "Europe/London", "event_type": "click"},
    {"event_id": 3, "local_time": "2024-06-15 23:00:00", "timezone": "Asia/Tokyo", "event_type": "purchase"},
    {"event_id": 4, "local_time": "2024-06-15 18:30:00", "timezone": "Asia/Kolkata", "event_type": "signup"},
    {"event_id": 5, "local_time": "2024-06-15 07:00:00", "timezone": "America/Los_Angeles", "event_type": "purchase"},
]

# Normalize to UTC
normalized = []
for event in events:
    local_dt = datetime.strptime(event["local_time"], "%Y-%m-%d %H:%M:%S")
    aware_local = local_dt.replace(tzinfo=ZoneInfo(event["timezone"]))
    utc_dt = aware_local.astimezone(timezone.utc)
    
    normalized.append({
        "event_id": event["event_id"],
        "event_type": event["event_type"],
        "local_time": event["local_time"],
        "source_timezone": event["timezone"],
        "utc_time": utc_dt.strftime("%Y-%m-%d %H:%M:%S"),
        "utc_date": utc_dt.strftime("%Y-%m-%d"),
        "utc_hour": utc_dt.hour
    })

pdf = pd.DataFrame(normalized)
print("Timezone-normalized events:")
print(pdf[["event_id", "local_time", "source_timezone", "utc_time"]].to_string(index=False))

# Write to Spark
spark.createDataFrame(pdf).write.format("delta").mode("overwrite").saveAsTable("techmart_dw.events_utc")
print(f"\n✅ Written {len(pdf)} events normalized to UTC")

## Project 2: Complete Date Dimension Generator

In [0]:
%sql
-- Production date dimension (2020-2030)
USE techmart_dw;
DROP TABLE IF EXISTS dim_date_complete;
CREATE TABLE dim_date_complete AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d AS full_date,
  YEAR(d) AS year,
  QUARTER(d) AS quarter,
  MONTH(d) AS month,
  date_format(d, 'MMMM') AS month_name,
  date_format(d, 'MMM') AS month_short,
  DAY(d) AS day_of_month,
  DAYOFWEEK(d) AS day_of_week,
  date_format(d, 'EEEE') AS day_name,
  WEEKOFYEAR(d) AS iso_week_number,
  DAYOFYEAR(d) AS day_of_year,
  CASE WHEN DAYOFWEEK(d) IN (1, 7) THEN true ELSE false END AS is_weekend,
  CASE WHEN MONTH(d) >= 7 THEN YEAR(d) + 1 ELSE YEAR(d) END AS fiscal_year,
  CASE WHEN MONTH(d) >= 7 THEN CEILING((MONTH(d) - 6) / 3.0) ELSE CEILING((MONTH(d) + 6) / 3.0) END AS fiscal_quarter,
  CONCAT(YEAR(d), '-Q', QUARTER(d)) AS year_quarter,
  CONCAT(YEAR(d), '-', LPAD(CAST(MONTH(d) AS STRING), 2, '0')) AS year_month,
  DAY(last_day(d)) AS days_in_month,
  CASE WHEN d = last_day(d) THEN true ELSE false END AS is_month_end,
  CASE WHEN d = last_day(d) AND MONTH(d) IN (3,6,9,12) THEN true ELSE false END AS is_quarter_end
FROM (
  SELECT date_add('2020-01-01', id-1) AS d
  FROM (SELECT explode(sequence(1, 4018)) AS id)
) dates;

SELECT COUNT(*) AS total_days, MIN(full_date) AS start_date, MAX(full_date) AS end_date
FROM dim_date_complete;

---
# 🏆 Section 9: Interview Questions

## Q1: How do you handle timezones in a global data platform?

1. **Ingest:** Convert all timestamps to UTC on arrival
2. **Store:** Always store as UTC in the data lake
3. **Process:** All pipeline logic operates on UTC
4. **Display:** Convert to user's local timezone only at the presentation layer
5. **Metadata:** Store the original timezone alongside UTC for audit

## Q2: Naive vs aware datetimes?

- **Naive:** No timezone info. `datetime(2024, 6, 15, 10, 0)` — 10 AM WHERE?
- **Aware:** Has timezone. `datetime(2024, 6, 15, 10, 0, tzinfo=timezone.utc)` — 10 AM UTC.
- **Rule:** Never use naive in production. Always attach timezone info.

## Q3: Late-arriving data with wrong timestamps?

1. Compare `event_time` vs `ingestion_time` — large gap = late arrival
2. Use `ingestion_time` for partitioning (ensures correct file placement)
3. Use `event_time` for business logic (correct chronological order)
4. Reprocess affected partitions if late data changes aggregates
5. Track late-arrival rate as a data quality metric

## Q4: Design a date dimension?

Key columns: date_key (INT YYYYMMDD), full_date, year, quarter, month, month_name,
day, day_of_week, day_name, is_weekend, is_holiday, fiscal_year, fiscal_quarter,
week_of_year, is_month_end. Generate for 10+ years. Update holidays annually.

## Q5: DST spring forward — what happens at 2 AM?

Clocks jump from 1:59:59 AM → 3:00:00 AM. The hour 2:00-2:59 doesn't exist.
If an event claims to happen at 2:30 AM during spring forward, it's invalid.
Options: reject it, shift to 3:30 AM, or flag for review.

## Q6: Seconds vs milliseconds epoch?

Count digits: 10 digits = seconds (1718444400), 13 digits = milliseconds (1718444400000).
Or check the resulting date: if `fromtimestamp(value)` gives a year > 2100, it's probably milliseconds.

## Q7: Fiscal calendar implementation?

Map each date to fiscal_year, fiscal_quarter, fiscal_period based on company rules.
Example: fiscal year starts July 1 → July 2024 = FY2025 Q1.
Build a lookup table (date → fiscal columns) and join in queries.

## Q8: event_time vs processing_time vs ingestion_time?

- **event_time:** When the event actually happened (from source system)
- **processing_time:** When your pipeline processed it (wall clock during ETL)
- **ingestion_time:** When it arrived in your system (landing timestamp)
- Use event_time for business logic, ingestion_time for partitioning, processing_time for debugging.

---
# 📗 Section 5: Timezone Handling Deep Dive

Timezone bugs are the #1 source of silent data corruption in DE pipelines.
The golden rule: **store everything in UTC, convert only for display.**

In [0]:
from datetime import datetime, timezone, timedelta
import zoneinfo  # Python 3.9+
from zoneinfo import ZoneInfo

# The three types of datetime objects:
# 1. Naive: no timezone info (DANGEROUS in DE!)
naive_dt = datetime(2024, 1, 15, 10, 30, 0)
print(f"Naive:   {naive_dt} (tzinfo={naive_dt.tzinfo})")

# 2. UTC-aware: always safe
utc_dt = datetime(2024, 1, 15, 10, 30, 0, tzinfo=timezone.utc)
print(f"UTC:     {utc_dt}")

# 3. Local timezone-aware
eastern = ZoneInfo("America/New_York")
eastern_dt = datetime(2024, 1, 15, 10, 30, 0, tzinfo=eastern)
print(f"Eastern: {eastern_dt}")

# Convert between timezones
pacific = ZoneInfo("America/Los_Angeles")
pacific_dt = eastern_dt.astimezone(pacific)
print(f"Pacific: {pacific_dt}")

# Convert to UTC (the ingestion pattern)
utc_from_eastern = eastern_dt.astimezone(timezone.utc)
print(f"→ UTC:   {utc_from_eastern}")

# ⚠️ DANGER: naive datetime comparison
try:
    diff = utc_dt - naive_dt  # TypeError!
except TypeError as e:
    print(f"\n⚠️  Cannot compare naive and aware: {e}")

# ✅ Safe: always work with aware datetimes
def make_utc(dt: datetime, assume_tz: str = "UTC") -> datetime:
    """Convert any datetime to UTC. Assumes timezone if naive."""
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=ZoneInfo(assume_tz))
    return dt.astimezone(timezone.utc)

print(f"\nNaive → UTC (assuming Eastern): {make_utc(naive_dt, 'America/New_York')}")

In [0]:
# --- DST (Daylight Saving Time) edge cases ---
eastern = ZoneInfo("America/New_York")

# Spring forward: 2024-03-10 2:00 AM → 3:00 AM (2:30 AM doesn't exist!)
# This is the "spring forward" gap
try:
    # 2:30 AM on spring-forward day — this time doesn't exist!
    ambiguous = datetime(2024, 3, 10, 2, 30, 0, tzinfo=eastern)
    print(f"Spring forward: {ambiguous}")
    # Python's zoneinfo handles this by folding
except Exception as e:
    print(f"Error: {e}")

# Fall back: 2024-11-03 1:30 AM happens TWICE
# fold=0 → first occurrence (EDT, UTC-4)
# fold=1 → second occurrence (EST, UTC-5)
fall_back_first = datetime(2024, 11, 3, 1, 30, 0, tzinfo=eastern, fold=0)
fall_back_second = datetime(2024, 11, 3, 1, 30, 0, tzinfo=eastern, fold=1)
print(f"Fall back (first):  {fall_back_first.astimezone(timezone.utc)}")
print(f"Fall back (second): {fall_back_second.astimezone(timezone.utc)}")
print(f"Difference: {fall_back_second.astimezone(timezone.utc) - fall_back_first.astimezone(timezone.utc)}")

# ✅ Production rule: always convert to UTC immediately on ingestion
# Never store local times in your data warehouse!

In [0]:
# --- Batch timezone conversion (Pandas) ---
import pandas as pd
import numpy as np

# Simulate events from multiple timezones
events = pd.DataFrame({
    "event_id": range(1, 11),
    "local_time": [
        "2024-01-15 09:00:00",
        "2024-01-15 14:30:00",
        "2024-01-15 22:00:00",
        "2024-03-10 02:30:00",  # Spring forward edge case
        "2024-11-03 01:30:00",  # Fall back edge case
        "2024-06-15 12:00:00",
        "2024-12-25 00:00:00",
        "2024-07-04 18:00:00",
        "2024-01-01 00:00:00",
        "2024-08-15 08:00:00",
    ],
    "source_timezone": [
        "America/New_York",
        "America/Los_Angeles",
        "Europe/London",
        "America/New_York",
        "America/New_York",
        "Asia/Tokyo",
        "Australia/Sydney",
        "America/Chicago",
        "UTC",
        "America/Denver",
    ],
})

def convert_to_utc_batch(df, time_col, tz_col):
    """Convert a column of local times to UTC using per-row timezone."""
    utc_times = []
    errors = []
    
    for idx, row in df.iterrows():
        try:
            local_dt = pd.Timestamp(row[time_col])
            tz = ZoneInfo(row[tz_col])
            aware_dt = local_dt.replace(tzinfo=tz)
            utc_dt = aware_dt.astimezone(timezone.utc)
            utc_times.append(pd.Timestamp(utc_dt))
        except Exception as e:
            utc_times.append(pd.NaT)
            errors.append({"idx": idx, "error": str(e)})
    
    return utc_times, errors

events["utc_time"], errs = convert_to_utc_batch(events, "local_time", "source_timezone")
print("Events normalized to UTC:")
print(events[["event_id", "local_time", "source_timezone", "utc_time"]].to_string())
if errs:
    print(f"\nErrors: {errs}")

## 5.2 Date Arithmetic & Business Logic

Common date calculations in DE pipelines.

In [0]:
from datetime import date
import calendar

# --- Business day calculations ---
def add_business_days(start_date: date, n_days: int,
                      holidays: list = None) -> date:
    """Add N business days to a date, skipping weekends and holidays."""
    holidays = set(holidays or [])
    current = start_date
    days_added = 0
    
    while days_added < n_days:
        current += timedelta(days=1)
        if current.weekday() < 5 and current not in holidays:  # Mon-Fri
            days_added += 1
    
    return current

# US holidays 2024 (simplified)
us_holidays_2024 = [
    date(2024, 1, 1),   # New Year's
    date(2024, 7, 4),   # Independence Day
    date(2024, 12, 25), # Christmas
]

order_date = date(2024, 12, 23)
delivery_date = add_business_days(order_date, 3, us_holidays_2024)
print(f"Order: {order_date}, Delivery (+3 biz days): {delivery_date}")

# --- Date dimension generation ---
def generate_date_dimension(start: date, end: date) -> pd.DataFrame:
    """Generate a complete date dimension table."""
    dates = pd.date_range(start, end, freq="D")
    df = pd.DataFrame({"date": dates})
    df["date_key"] = df["date"].dt.strftime("%Y%m%d").astype(int)
    df["year"] = df["date"].dt.year
    df["quarter"] = df["date"].dt.quarter
    df["month"] = df["date"].dt.month
    df["month_name"] = df["date"].dt.strftime("%B")
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["day_of_week"] = df["date"].dt.dayofweek  # 0=Mon, 6=Sun
    df["day_name"] = df["date"].dt.strftime("%A")
    df["is_weekend"] = df["day_of_week"] >= 5
    df["is_month_start"] = df["date"].dt.is_month_start
    df["is_month_end"] = df["date"].dt.is_month_end
    df["is_quarter_start"] = df["date"].dt.is_quarter_start
    df["is_quarter_end"] = df["date"].dt.is_quarter_end
    df["fiscal_year"] = df["year"].where(df["month"] < 10, df["year"] + 1)
    df["fiscal_quarter"] = ((df["month"] - 10) % 12 // 3 + 1)
    return df

dim_date = generate_date_dimension(date(2024, 1, 1), date(2024, 3, 31))
print(f"\nDate dimension: {len(dim_date)} rows")
print(dim_date.head(5).to_string())

---
# 📗 Section 6: Practice Exercises

In [0]:
# ============================================================
# EXERCISE 1: SLA Breach Detection
# ============================================================
# Given orders with created_at and completed_at timestamps,
# detect SLA breaches (orders taking > 2 business days)

orders = pd.DataFrame({
    "order_id": range(1, 8),
    "created_at": pd.to_datetime([
        "2024-01-15 09:00:00+00:00",
        "2024-01-15 14:00:00+00:00",
        "2024-01-12 16:00:00+00:00",  # Friday afternoon
        "2024-01-15 10:00:00+00:00",
        "2024-01-16 08:00:00+00:00",
        "2024-01-15 11:00:00+00:00",
        "2024-01-15 09:30:00+00:00",
    ]),
    "completed_at": pd.to_datetime([
        "2024-01-16 10:00:00+00:00",  # 1 day — OK
        "2024-01-18 09:00:00+00:00",  # 3 days — BREACH
        "2024-01-16 10:00:00+00:00",  # Fri→Tue = 1 biz day — OK
        None,                          # Not completed — check
        "2024-01-17 15:00:00+00:00",  # 1 day — OK
        "2024-01-19 10:00:00+00:00",  # 4 days — BREACH
        "2024-01-17 09:00:00+00:00",  # 2 days — OK (boundary)
    ]),
})

SLA_HOURS = 48  # 2 business days ≈ 48 hours

orders["duration_hours"] = (
    (orders["completed_at"] - orders["created_at"])
    .dt.total_seconds() / 3600
).round(1)

orders["sla_status"] = orders.apply(lambda r: (
    "not_completed" if pd.isna(r["completed_at"])
    else "breach" if r["duration_hours"] > SLA_HOURS
    else "ok"
), axis=1)

print("SLA Analysis:")
print(orders[["order_id", "duration_hours", "sla_status"]].to_string(index=False))
print(f"\nBreaches: {(orders['sla_status'] == 'breach').sum()}")
print(f"Not completed: {(orders['sla_status'] == 'not_completed').sum()}")

In [0]:
# ============================================================
# EXERCISE 2: Time-Series Gap Detection
# ============================================================
# Detect missing time periods in event data

hourly_events = pd.DataFrame({
    "hour": pd.date_range("2024-01-15 00:00", periods=24, freq="h"),
    "event_count": [100, 95, 88, 72, 65, 70, 120, 250, 380, 420,
                    410, 395, 380, 370, 360, 340, 320, 290, 260, 230,
                    200, 170, 140, 110],
})
# Simulate gaps by removing some hours
hourly_events = hourly_events[~hourly_events.index.isin([5, 6, 15])]

def detect_time_gaps(df, time_col, expected_freq="h"):
    """Detect missing time periods in a time series."""
    full_range = pd.date_range(
        df[time_col].min(), df[time_col].max(), freq=expected_freq
    )
    missing = full_range[~full_range.isin(df[time_col])]
    
    print(f"Expected periods: {len(full_range)}")
    print(f"Actual periods:   {len(df)}")
    print(f"Missing periods:  {len(missing)}")
    if len(missing) > 0:
        print(f"Missing times:    {missing.tolist()}")
    return missing

gaps = detect_time_gaps(hourly_events, "hour")

---
# 📗 Section 5: Timezone Handling -- The #1 Source of DE Bugs

## The Timezone Problem

```
Event happens at: 2024-03-15 10:00:00 UTC
Stored as:        2024-03-15 10:00:00 (no timezone info!)
Analyst in NYC:   "This happened at 6 AM" (UTC-4)
Analyst in London:"This happened at 10 AM" (UTC+0)
Analyst in Tokyo: "This happened at 7 PM" (UTC+9)

Same timestamp, three different interpretations!
```

## Best Practices

1. **Always store in UTC** -- convert to local time only for display
2. **Always include timezone info** -- use timezone-aware datetimes
3. **Be explicit about conversions** -- document every timezone conversion
4. **Use ISO 8601 format** -- `2024-03-15T10:00:00Z` (the Z means UTC)

## Common Timezone Operations

```python
from datetime import datetime, timezone
import pytz

# Create timezone-aware datetime
utc_now = datetime.now(timezone.utc)
print(utc_now)  # 2024-03-15 10:00:00+00:00

# Convert to specific timezone
eastern = pytz.timezone("America/New_York")
eastern_time = utc_now.astimezone(eastern)
print(eastern_time)  # 2024-03-15 06:00:00-04:00

# Parse timezone-aware string
from dateutil import parser
dt = parser.parse("2024-03-15T10:00:00Z")
print(dt.tzinfo)  # UTC

# Convert naive datetime to UTC (DANGEROUS -- assumes local time)
naive = datetime(2024, 3, 15, 10, 0, 0)
utc = pytz.utc.localize(naive)  # Assume it's UTC
```

## Daylight Saving Time (DST) Gotchas

```python
# DST transition: 2024-03-10 02:00 AM EST -> 03:00 AM EDT
eastern = pytz.timezone("America/New_York")

# This time doesn't exist! (clocks spring forward)
ambiguous_time = datetime(2024, 3, 10, 2, 30, 0)
try:
    eastern.localize(ambiguous_time, is_dst=None)  # Raises AmbiguousTimeError
except pytz.exceptions.NonExistentTimeError:
    print("This time doesn't exist due to DST!")

# Safe approach: always work in UTC, convert only for display
```

In [0]:
# Datetime patterns for data engineering
from datetime import datetime, timezone, timedelta
import pytz

print("Datetime Patterns for Data Engineering")
print("=" * 60)

# Pattern 1: Always use UTC for storage
utc_now = datetime.now(timezone.utc)
print(f"\n1. UTC timestamp: {utc_now.isoformat()}")

# Pattern 2: Convert for display
eastern = pytz.timezone("America/New_York")
eastern_time = utc_now.astimezone(eastern)
print(f"2. Eastern time: {eastern_time.strftime('%Y-%m-%d %H:%M:%S %Z')}")

# Pattern 3: Date arithmetic
today = datetime.now(timezone.utc).date()
yesterday = today - timedelta(days=1)
last_week = today - timedelta(weeks=1)
print(f"\n3. Date arithmetic:")
print(f"   Today: {today}")
print(f"   Yesterday: {yesterday}")
print(f"   Last week: {last_week}")

# Pattern 4: Parse various formats
from dateutil import parser as dateutil_parser
formats_to_parse = [
    "2024-03-15",
    "2024-03-15T10:00:00Z",
    "March 15, 2024",
    "15/03/2024",
    "2024-03-15 10:00:00+05:30",
]
print("\n4. Parsing various formats:")
for fmt in formats_to_parse:
    try:
        parsed = dateutil_parser.parse(fmt)
        print(f"   '{fmt}' -> {parsed.isoformat()}")
    except Exception as e:
        print(f"   '{fmt}' -> ERROR: {e}")

# Pattern 5: Truncation (for grouping)
print("\n5. Date truncation (for GROUP BY):")
ts = datetime(2024, 3, 15, 14, 37, 22)
print(f"   Original: {ts}")
print(f"   By hour:  {ts.replace(minute=0, second=0, microsecond=0)}")
print(f"   By day:   {ts.replace(hour=0, minute=0, second=0, microsecond=0)}")
print(f"   By month: {ts.replace(day=1, hour=0, minute=0, second=0, microsecond=0)}")


---
# ✅ Notebook Complete!

**What was covered:**
- Python datetime: creation, arithmetic, formatting, parsing
- Timezones: UTC standard, zoneinfo, DST edge cases (spring forward, fall back)
- Epoch timestamps: auto-detection of seconds/milliseconds/microseconds
- Parsing messy dates: dateutil, pd.to_datetime with errors='coerce'
- PySpark: date functions, time-based window aggregation
- Date partitioning: choosing granularity, partition pruning
- 2 mini projects: Timezone-Aware Processing, Complete Date Dimension
- 8 interview questions

**Next:** Notebook 15 — JSON for Data Engineering

---